In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
OBO Ontology Graph Parser adapted for Google Colab.

This script parses OBO files and extracts the ontology graph structure,
focusing on 'is_a' relationships to build parent-child hierarchies.

Usage in Colab:
1. Upload your OBO file (e.g., 'cl.obo') to your Colab environment.
   You can do this by clicking the folder icon on the left sidebar ->
   'Files' tab -> 'Upload to session storage'.
2. Run the `main` function with your input OBO file name and desired output JSON file name.
   Example: main('cl.obo', 'cl_graph.json')
"""

import re
import json
from pathlib import Path
from typing import Dict, List, Set


class OBOGraphParser:
    """Parser for OBO ontology graph structure."""

    def __init__(self):
        self.terms: Dict[str, Dict] = {}
        self.graph: Dict[str, List[str]] = {}  # term_id -> [parent_ids]
        self.stats = {
            'total_terms': 0,
            'terms_with_parents': 0,
            'total_relationships': 0
        }

    def parse_file(self, filepath: str) -> None:
        """Parse an OBO file and extract graph structure."""
        print(f"Processing {filepath}...")

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
        except FileNotFoundError:
            print(f"Error: File not found at {filepath}. Please ensure the file is uploaded or the path is correct.")
            return
        except Exception as e:
            print(f"Error reading file {filepath}: {e}")
            return

        # Split into term blocks
        blocks = self._split_into_blocks(content)

        for block in blocks:
            if not block.strip():
                continue

            lines = [line.strip() for line in block.strip().split('\n')]
            if not lines:
                continue

            # Check if the block starts with a valid block type, e.g., [Term]
            if not lines[0].startswith('[') or not lines[0].endswith(']'):
                continue # Skip malformed blocks

            block_type = lines[0].strip('[]')

            if block_type == 'Term':
                self._parse_term_block(lines[1:])

        print(f"✓ Successfully processed {filepath}")

    def _split_into_blocks(self, content: str) -> List[str]:
        """Split OBO content into blocks based on [Term], [Typedef], etc."""
        # This regex looks for lines starting with '[', followed by one or more
        # characters that are not ']', and ending with ']', followed by a newline.
        # It then splits the content based on these block headers.
        block_pattern = re.compile(r'^\[[^\]]+\]\n', re.MULTILINE)

        # Find all block headers and their positions
        matches = list(block_pattern.finditer(content))

        if not matches:
            # If no explicit blocks are found, treat the whole content as one block
            return [content]

        blocks = []
        for i, match in enumerate(matches):
            start = match.start()
            # The end of the current block is the start of the next block,
            # or the end of the content if it's the last block.
            end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
            blocks.append(content[start:end])

        return blocks

    def _parse_term_block(self, lines: List[str]) -> None:
        """Parse a [Term] block and extract term info and relationships."""
        term_id = None
        name = ""
        parents = []
        is_obsolete = False

        for line in lines:
            if not line or line.startswith('!'):  # Skip empty lines or comments
                continue

            if ':' not in line:  # Skip lines not containing key-value pairs
                continue

            key, value = line.split(':', 1)
            key = key.strip()
            value = value.strip()

            if key == 'id':
                # Only process CL (Cell Ontology) terms
                if value.startswith('CL:'):
                    term_id = value
            elif key == 'name':
                name = value
            elif key == 'is_a':
                # Extract parent ID from is_a relationship
                # Format examples:
                # "is_a: CL:0000021 {is_inferred="true"} ! female germ cell"
                # "is_a: CL:0000021 ! female germ cell"
                # "is_a: CL:0000021"
                parent_match = re.match(r'([A-Z_]+:\d+)', value)  # Updated regex to include underscore
                if parent_match:
                    parent_id = parent_match.group(1)
                    # Only include CL (Cell Ontology) terms
                    if parent_id.startswith('CL:'):
                        parents.append(parent_id)
            elif key == 'is_obsolete':
                is_obsolete = value.lower() == 'true'

        # Store term info if we have a valid CL ID and it's not obsolete
        if term_id and term_id.startswith('CL:') and not is_obsolete:
            self.terms[term_id] = {
                'name': name,
                'parents': parents
            }

            # Store in graph structure
            self.graph[term_id] = parents

            self.stats['total_terms'] += 1
            if parents:
                self.stats['terms_with_parents'] += 1
                self.stats['total_relationships'] += len(parents) == 'true'

        # Store term info if we have a valid ID and it's not obsolete
        if term_id and not is_obsolete:
            self.terms[term_id] = {
                'name': name,
                'parents': parents
            }

            # Store in graph structure
            self.graph[term_id] = parents

            self.stats['total_terms'] += 1
            if parents:
                self.stats['terms_with_parents'] += 1
                self.stats['total_relationships'] += len(parents)

    def save_graph(self, output_file: str) -> None:
        """Save the ontology graph to a JSON file."""
        graph_data = {
            'metadata': {
                'total_terms': self.stats['total_terms'],
                'terms_with_parents': self.stats['terms_with_parents'],
                'total_relationships': self.stats['total_relationships']
            },
            'terms': self.terms,
            'graph': self.graph
        }

        try:
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(graph_data, f, indent=2, ensure_ascii=False)

            print(f"\n✓ Successfully saved ontology graph to {output_file}")
            self._print_stats()

        except Exception as e:
            print(f"✗ Error saving graph file: {e}")

    def _print_stats(self) -> None:
        """Print processing statistics."""
        print(f"\nProcessing Statistics:")
        print(f"  Total terms: {self.stats['total_terms']}")
        print(f"  Terms with parents: {self.stats['terms_with_parents']}")
        print(f"  Total is_a relationships: {self.stats['total_relationships']}")

    def print_sample_hierarchy(self, n: int = 5) -> None:
        """Print a sample of the hierarchy structure."""
        print(f"\nSample hierarchy (first {n} terms):")
        count = 0
        for term_id, term_data in self.terms.items():
            if count >= n:
                break

            print(f"  {term_id}: {term_data['name']}")
            if term_data['parents']:
                for parent_id in term_data['parents']:
                    parent_name = self.terms.get(parent_id, {}).get('name', 'Unknown')
                    print(f"    ↳ is_a: {parent_id} ({parent_name})")
            else:
                print(f"    ↳ (root term or no 'is_a' parents found)")
            print()
            count += 1

    def find_roots(self) -> List[str]:
        """Find root terms (terms with no 'is_a' parents)."""
        roots = []
        for term_id, parents in self.graph.items():
            if not parents: # A term is a root if it has no 'is_a' parents in the parsed graph
                roots.append(term_id)
        return roots

    def find_leaves(self) -> List[str]:
        """Find leaf terms (terms that are not 'is_a' parents of any other term)."""
        all_parents_in_graph = set()
        for parents_list in self.graph.values():
            all_parents_in_graph.update(parents_list)

        leaves = []
        for term_id in self.graph.keys():
            # A term is a leaf if it exists in the graph but is not listed as a parent for any other term
            if term_id not in all_parents_in_graph:
                leaves.append(term_id)
        return leaves


def main(obo_file_path: str, output_json_path: str, sample_terms_to_display: int = 50):
    """
    Main function to parse an OBO file and save its graph structure.
    Designed for use in Google Colab.

    Args:
        obo_file_path (str): The path to the input OBO file.
        output_json_path (str): The desired path for the output JSON file.
        sample_terms_to_display (int): Number of sample terms to display in the output.
    """
    try:
        parser_obj = OBOGraphParser()

        # Process the input file
        parser_obj.parse_file(obo_file_path)

        if not parser_obj.terms:
            print("No terms found in the input file or parsing failed.")
            return

        # Show sample hierarchy
        parser_obj.print_sample_hierarchy(sample_terms_to_display)

        # Find and display roots and leaves
        roots = parser_obj.find_roots()
        leaves = parser_obj.find_leaves()

        print(f"\nGraph Analysis:")
        print(f"  Root terms (no 'is_a' parents in the parsed graph): {len(roots)}")
        print(f"  Leaf terms (no 'is_a' children in the parsed graph): {len(leaves)}")

        # Save the graph
        parser_obj.save_graph(output_json_path)

        print(f"\n✓ Graph parsing complete! Output saved to: {output_json_path}")

        # Instructions for downloading the file in Colab
        print(f"\nTo download the output file '{output_json_path}' in Colab, run the following in a new cell:")
        print(f"  from google.colab import files")
        print(f"  files.download('{output_json_path}')")

    except Exception as e:
        print(f"✗ An unexpected error occurred: {e}")


In [ ]:
main('cl.obo', 'cl_graph.json', sample_terms_to_display=10)


Processing cl.obo...
✓ Successfully processed cl.obo

Sample hierarchy (first 10 terms):
  CL:0000000: cell
    ↳ (root term or no 'is_a' parents found)

  CL:0000001: primary cultured cell
    ↳ is_a: CL:0000010 (cultured cell)

  CL:0000005: neural crest derived fibroblast
    ↳ is_a: CL:0000057 (fibroblast)

  CL:0000006: neuronal receptor cell
    ↳ is_a: CL:0000101 (sensory neuron)
    ↳ is_a: CL:0000197 (sensory receptor cell)

  CL:0000007: early embryonic cell (metazoa)
    ↳ is_a: CL:0002321 (embryonic cell (metazoa))

  CL:0000008: migratory cranial neural crest cell
    ↳ is_a: CL:0000333 (migratory neural crest cell)

  CL:0000010: cultured cell
    ↳ is_a: CL:0000578 (experimentally modified cell in vitro)

  CL:0000011: migratory trunk neural crest cell
    ↳ is_a: CL:0000333 (migratory neural crest cell)

  CL:0000014: germ line stem cell
    ↳ is_a: CL:0000034 (stem cell)
    ↳ is_a: CL:0000039 (germ line cell)

  CL:0000015: male germ cell
    ↳ is_a: CL:0000586 (germ 

In [ ]:
import json
import numpy as np
import networkx as nx
from tqdm import tqdm
import pickle

class OntologySimilarityCalculator:
    """Calculate and save PageRank-based similarities between cell types"""

    def __init__(self, alpha=0.9, threshold=1e-4):
        self.alpha = alpha
        self.threshold = threshold
        self.graph = None
        self.cl_id_to_name = {}
        self.name_to_cl_id = {}
        self.similarities = None

    def load_ontology_graph(self, json_file_path):
        """Load ontology graph from JSON file created by OBO parser"""
        with open(json_file_path, 'r') as f:
            data = json.load(f)

        # Create mappings between CL IDs and normalized names
        for cl_id, term_data in data['terms'].items():
            name = self.normalize_cell_type(term_data['name'])
            self.cl_id_to_name[cl_id] = name
            self.name_to_cl_id[name] = cl_id

        # Build NetworkX graph
        edge_list = []
        for child_id, parent_ids in data['graph'].items():
            for parent_id in parent_ids:
                edge_list.append((parent_id, child_id))  # parent -> child

        self.graph = nx.DiGraph(edge_list)
        print(f"Loaded ontology graph with {len(self.cl_id_to_name)} terms")

    def normalize_cell_type(self, cell_type_name):
        """Normalize cell type name to match extractor normalization"""
        if not cell_type_name:
            return "unknown"

        normalized = cell_type_name.lower().strip()

        # Remove common suffixes - expanded list for better normalization
        suffixes_to_remove = ["cell", "cells", "celltype", "cell type"]
        for suffix in suffixes_to_remove:
            if normalized.endswith(" " + suffix):
                normalized = normalized[:-len(" " + suffix)]
            # Also handle cases where the suffix is at the end without space
            elif normalized.endswith(suffix) and len(normalized) > len(suffix):
                # Check if removing suffix doesn't leave us with empty string
                potential_result = normalized[:-len(suffix)]
                if potential_result.strip():
                    normalized = potential_result

        # Clean up any remaining whitespace
        normalized = normalized.strip()

        # Handle edge case where we might end up with empty string
        if not normalized:
            return "unknown"

        return normalized

    def compute_pagerank_similarities(self):
        """Compute PageRank similarities between all cell types"""
        cl_ids = list(self.cl_id_to_name.keys())
        n_terms = len(cl_ids)

        # Create mapping from CL_ID to matrix index
        cl_id_to_idx = {cl_id: i for i, cl_id in enumerate(cl_ids)}

        # Initialize similarity matrix
        similarities = np.zeros((n_terms, n_terms), dtype=np.float32)

        print("Computing PageRank similarities...")
        for i, source_cl_id in enumerate(tqdm(cl_ids)):
            if source_cl_id not in self.graph:
                continue

            # Set personalization vector
            personalization = {cl_id: (cl_id == source_cl_id) for cl_id in cl_ids if cl_id in self.graph}

            try:
                # Compute PageRank
                ppr = nx.pagerank(self.graph, alpha=self.alpha, personalization=personalization)

                # Store similarities with transformation
                for target_cl_id, score in ppr.items():
                    if target_cl_id in cl_id_to_idx:
                        j = cl_id_to_idx[target_cl_id]
                        # Apply threshold transformation
                        if score < self.threshold:
                            similarities[i, j] = 1
                        else:
                            similarities[i, j] = np.log2(score * (1 / self.threshold) + 1)

            except Exception as e:
                print(f"Error computing PageRank for {source_cl_id}: {e}")
                continue

        # Create name-based similarity mapping for easy lookup
        # This uses the normalized names (without "cell") as keys
        name_similarities = {}
        for i, source_cl_id in enumerate(cl_ids):
            source_name = self.cl_id_to_name[source_cl_id]  # Already normalized
            name_similarities[source_name] = {}

            for j, target_cl_id in enumerate(cl_ids):
                target_name = self.cl_id_to_name[target_cl_id]  # Already normalized
                name_similarities[source_name][target_name] = float(similarities[i, j])

        self.similarities = name_similarities
        print(f"Created similarity matrix with normalized names (examples: {list(name_similarities.keys())[:5]})")
        return name_similarities

    def save_similarities(self, output_path):
        """Save computed similarities to file"""
        if self.similarities is None:
            raise ValueError("No similarities computed. Run compute_pagerank_similarities first.")

        with open(output_path, 'wb') as f:
            pickle.dump(self.similarities, f)

        print(f"Saved similarities to {output_path}")
        print(f"Matrix contains {len(self.similarities)} cell types")

    def load_similarities(self, similarity_file_path):
        """Load precomputed similarities"""
        with open(similarity_file_path, 'rb') as f:
            self.similarities = pickle.load(f)
        return self.similarities

    def get_similarity(self, cell_type1, cell_type2):
        """Get similarity between two cell types (both should be normalized names)"""
        if self.similarities is None:
            raise ValueError("No similarities loaded. Run compute_pagerank_similarities or load_similarities first.")

        # Normalize the input names in case they weren't already
        norm_type1 = self.normalize_cell_type(cell_type1)
        norm_type2 = self.normalize_cell_type(cell_type2)

        return self.similarities.get(norm_type1, {}).get(norm_type2, 0.0)

    def print_sample_similarities(self, cell_type_name, top_k=10):
        """Print top similar cell types for a given cell type"""
        if self.similarities is None:
            raise ValueError("No similarities loaded.")

        norm_name = self.normalize_cell_type(cell_type_name)

        if norm_name not in self.similarities:
            print(f"Cell type '{norm_name}' not found in similarities.")
            available = list(self.similarities.keys())[:10]
            print(f"Available cell types (first 10): {available}")
            return

        # Get similarities for this cell type and sort by score
        sims = self.similarities[norm_name]
        sorted_sims = sorted(sims.items(), key=lambda x: x[1], reverse=True)

        print(f"Top {top_k} similar cell types to '{norm_name}':")
        for i, (target_name, score) in enumerate(sorted_sims[:top_k]):
            print(f"  {i+1:2d}. {target_name:<30} (similarity: {score:.4f})")


def create_similarity_file(obo_json_path, output_similarity_path):
    """Helper function to create similarity file from OBO JSON"""
    calculator = OntologySimilarityCalculator()
    calculator.load_ontology_graph(obo_json_path)
    calculator.compute_pagerank_similarities()
    calculator.save_similarities(output_similarity_path)

    # Show some examples of the normalized names
    sample_names = list(calculator.similarities.keys())[:10]
    print(f"Sample normalized cell type names: {sample_names}")

    return calculator


# Example usage:
if __name__ == "__main__":
    # Step 1: Create similarity file (run once)
    calculator = create_similarity_file("cl_graph.json", "/content/drive/MyDrive/cell_type_similarities.pkl")

    # Step 2: Test the similarity lookup
    # Example: show similarities for "T helper" (assuming this exists in your ontology)
    calculator.print_sample_similarities("T helper cell", top_k=5)

Loaded ontology graph with 2874 terms
Computing PageRank similarities...


100%|██████████| 2874/2874 [01:00<00:00, 47.56it/s]


Created similarity matrix with normalized names (examples: ['cell', 'primary cultured', 'neural crest derived fibroblast', 'neuronal receptor', 'early embryonic cell (metazoa)'])
Saved similarities to /content/drive/MyDrive/cell_type_similarities.pkl
Matrix contains 2874 cell types
Sample normalized cell type names: ['cell', 'primary cultured', 'neural crest derived fibroblast', 'neuronal receptor', 'early embryonic cell (metazoa)', 'migratory cranial neural crest', 'cultured', 'migratory trunk neural crest', 'germ line stem', 'male germ']
Cell type 't helper' not found in similarities.
Available cell types (first 10): ['cell', 'primary cultured', 'neural crest derived fibroblast', 'neuronal receptor', 'early embryonic cell (metazoa)', 'migratory cranial neural crest', 'cultured', 'migratory trunk neural crest', 'germ line stem', 'male germ']


In [ ]:
import pickle
import os

def print_pkl_file_contents(file_path):
    """
    Loads and prints the contents of a .pkl file.

    Args:
        file_path (str): The path to the .pkl file.
    """
    if not os.path.exists(file_path):
        print(f"Error: The file '{file_path}' does not exist.")
        return

    try:
        with open(file_path, 'rb') as f:
            # Load the object from the pickle file
            data = pickle.load(f)
            print(f"Contents of '{file_path}':")
            print(data)
    except pickle.UnpicklingError as e:
        print(f"Error: Could not unpickle the file '{file_path}'. It might be corrupted or not a valid pickle file.")
        print(f"Details: {e}")
    except Exception as e:
        print(f"An unexpected error occurred while processing '{file_path}': {e}")

# --- Example Usage ---
if __name__ == "__main__":

    file_name = "cell_type_similarities.pkl"
    print_pkl_file_contents(file_name)




Buffered data was truncated after reaching the output size limit.